In [ ]:
from pathlib import Path
import json
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def get_pairwise_similarity_between_two_sets(embeddings_1, embeddings_2):
    # Convert to numpy arrays
    embeddings_1 = np.asarray(embeddings_1)
    embeddings_2 = np.asarray(embeddings_2)
    
    # Compute norms for all embeddings at once
    norms_1 = np.linalg.norm(embeddings_1, axis=1)
    norms_2 = np.linalg.norm(embeddings_2, axis=1)
    
    # Compute all similarities at once using matrix multiplication
    similarities = np.dot(embeddings_1, embeddings_2.T) / np.outer(norms_1, norms_2)
    
    # Flatten to match original output format
    return similarities.flatten()

In [ ]:
# Adapter für fixes Format [ [[vec],...], [[vec],...], ... ]

def load_embedding_data(file_path: str):
    path = Path(file_path)
    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)
    return [{"embeddings": question_embeddings} for question_embeddings in raw_data]

In [ ]:
def model_label_from_file(path_str: str):
    name = Path(path_str).stem
    if name.startswith("embeddings_"):
        name = name[len("embeddings_"):]
    return name


def main_heatmap_across_models(file_paths):
    
    selected_files = list(file_paths)

    selected_model_names = [model_label_from_file(p) for p in selected_files]
    model_similarities = {model_name: {} for model_name in selected_model_names}

    model_data_cache = {}
    for model_name, model_file in tqdm(
        list(zip(selected_model_names, selected_files)),
        desc="Loading model data",
    ):
        model_data_cache[model_name] = load_embedding_data(model_file)

    for i, model_name in enumerate(tqdm(selected_model_names, desc="Calculating similarities")):
        model_data = model_data_cache[model_name]
        for model_name_2 in selected_model_names[: i + 1]:
            model_data_2 = model_data_cache[model_name_2]

            all_similarities = []
            for d, d_2 in zip(model_data, model_data_2):
                similarities = get_pairwise_similarity_between_two_sets(
                    d["embeddings"],
                    d_2["embeddings"],
                )
                if len(similarities) > 0:
                    all_similarities.extend(similarities)

            model_similarities[model_name][model_name_2] = (
                float(np.mean(all_similarities)) if len(all_similarities) > 0 else np.nan
            )

    similarity_matrix = np.full((len(selected_model_names), len(selected_model_names)), np.nan)
    for i, model_name in enumerate(selected_model_names):
        for j, model_name_2 in enumerate(selected_model_names[: i + 1]):
            similarity_matrix[i, j] = model_similarities[model_name][model_name_2]

    plt.figure(figsize=(12, 12))
    mask = np.triu(np.ones_like(similarity_matrix, dtype=bool), k=1)
    sns.heatmap(
        similarity_matrix,
        annot=True,
        fmt=".3f",
        cmap="YlOrRd",
        xticklabels=selected_model_names,
        yticklabels=selected_model_names,
        square=True,
        mask=mask,
        vmin=0.5,
        vmax=1.0,
        cbar=False,
    )

    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    for text_obj in plt.gca().texts:
        text = text_obj.get_text()
        if text.startswith("0"):
            text_obj.set_text(text[1:])

    plt.tight_layout()
    plt.show()

# alle Embedding-Dateien einsammeln
all_model_files = sorted([str(p) for p in Path("./replikation").glob("embeddings_*.json")])
#all_model_files = sorted([str(p) for p in Path("./eigenanteil").glob("embeddings_*.json")])

print("Gefundene Dateien:", len(all_model_files))
for p in all_model_files:
    print("-", p)

main_heatmap_across_models(all_model_files)